# Search-16 : Contraintes Souples - Soft CSP

**Navigation** : [<< Search-15 CSP-Hybridization](Search-15-CSP-Hybridization.ipynb) | [Index](../README.md) | [Search-17 CSP-Temporal >>](Search-17-CSP-Temporal.ipynb)

## Problemes de Satisfaction de Contraintes Souples

Ce notebook explore les extensions du formalisme CSP classique pour gerer les **preferences** et les **contraintes violables**. Contrairement au CSP classique (binaire : satisfait/non satisfait), les **Soft CSP** attribuent des degres de satisfaction.

### Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Comprendre** les limitations du CSP classique binaire (Bloom : comprendre)
2. **Decouvrir** le framework semiring-based CSP unifie (Bloom : comprendre)
3. **Implementer** des Fuzzy CSP avec degres d'appartenance (Bloom : appliquer)
4. **Resoudre** des Weighted CSP avec couts de violation (Bloom : appliquer)
5. **Appliquer** les contraintes souples aux problemes de preferences (Bloom : analyser)

### Prerequis
- Search-6 : formalisme CSP classique
- Search-7 : propagation de contraintes
- Bases d'algebre (semirings)

### Duree estimee : 60 minutes

---

## 1. Introduction : Pourquoi les contraintes souples ? (~10 min)

Dans le CSP classique, une contrainte est soit **satisfaite** soit **violee**. Cette vision binaire est limitee dans de nombreuses situations reelles :

### Exemples de problemes

1. **Planning de vacances** : "Je prefererais partir en juillet, mais aout est acceptable"
2. **Recommandation** : "L'utilisateur prefere les films d'action, mais peut regarder autre chose"
3. **Allocation de ressources** : "Le budget ideal est 1000 EUR, mais 1200 EUR est tolerable"

### Taxonomie des Soft CSP

| Type | Valeurs | Combinaison | Application |
|------|---------|-------------|-------------|
| **Fuzzy CSP** | [0, 1] | min/max | Preferences floues |
| **Weighted CSP** | Reels positifs | somme | Couts de violation |
| **Probabilistic CSP** | [0, 1] | produit | Incertitude |
| **Valued CSP** | Semi-anneau | operateur * | General |

### Approche unifiee : Semiring-based CSP

Tous ces types peuvent etre unifies avec la notion de **semi-anneau** $(S, +, \times)$ :
- $S$ : ensemble des valeurs de preference
- $+$ : operateur de combinaison (agrege les contraintes)
- $\times$ : operateur de projection (propagation)

In [ ]:
# Installation des dependances
import subprocess
import sys

def install_if_missing(package):
    try:
        __import__(package.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

install_if_missing('ortools')
install_if_missing('matplotlib')
install_if_missing('numpy')

from ortools.sat.python import cp_model
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass
from abc import ABC, abstractmethod

print("Dependances pretes.")

---

## 2. Semiring-based CSP : Framework Algebrique (~15 min)

### Definition formelle

Un **semi-anneau** (semiring) est une structure algebrique $(S, +, \times)$ avec :
- $S$ : ensemble non vide
- $+$ : operateur commutatif, associatif, element neutre $\bot$
- $\times$ : operateur associatif, element neutre $\top$
- Distributivite de $\times$ sur $+$

### Exemples de semi-anneaux pour CSP

| Semi-anneau | S | + (combinaison) | x (projection) | Application |
|-------------|---|-----------------|----------------|-------------|
| **Booleen** | {True, False} | or | and | CSP classique |
| **Fuzzy** | [0, 1] | max | min | Preferences |
| **Weighted** | R+ U {+inf} | min | + | Couts |
| **Probabilistic** | [0, 1] | max | x | Incertitude |

### Proprietes cles

- **$ot$** : pire valeur (False, 0, +inf, 0)
- **$	op$** : meilleure valeur (True, 1, 0, 1)
- L'ordre est defini par : $a \leq_S b$ ssi $a + b = b$

In [ ]:
@dataclass
class Semiring:
    """
    Representation d'un semi-anneau pour Soft CSP.
    """
    name: str
    values: Any  # Description de l'ensemble
    combine: callable  # Operateur +
    project: callable  # Operateur x
    bottom: Any  # Element neutre de + (pire valeur)
    top: Any  # Element neutre de x (meilleure valeur)

# Semi-anneau Booleen (CSP classique)
BooleanSemiring = Semiring(
    name="Boolean",
    values="{True, False}",
    combine=lambda a, b: a or b,
    project=lambda a, b: a and b,
    bottom=False,
    top=True
)

# Semi-anneau Fuzzy
FuzzySemiring = Semiring(
    name="Fuzzy",
    values="[0, 1]",
    combine=max,  # max pour aggregation
    project=min,  # min pour combinaison
    bottom=0.0,
    top=1.0
)

# Semi-anneau Weighted
WeightedSemiring = Semiring(
    name="Weighted",
    values="R+ U {+inf}",
    combine=min,  # min pour choix du meilleur cout
    project=lambda a, b: a + b,  # addition pour agregation des couts
    bottom=float('inf'),  # Pire : cout infini
    top=0.0  # Meilleur : cout nul
)

print("Semi-anneaux definis :")
for sr in [BooleanSemiring, FuzzySemiring, WeightedSemiring]:
    print(f"  {sr.name}: bottom={sr.bottom}, top={sr.top}")

In [ ]:
class SoftCSP:
    """
    Soft CSP generique base sur un semi-anneau.
    """
    
    def __init__(self, semiring: Semiring):
        self.semiring = semiring
        self.variables = {}
        self.constraints = []  # Liste de (vars, fonction_eval)
    
    def add_variable(self, name: str, domain: List[Any]):
        """Ajoute une variable avec son domaine."""
        self.variables[name] = domain
    
    def add_constraint(self, involved_vars: List[str], evaluator: callable):
        """
        Ajoute une contrainte souple.
        evaluator : fonction(assignments) -> valeur semi-anneau
        """
        self.constraints.append((involved_vars, evaluator))
    
    def evaluate_assignment(self, assignment: Dict[str, Any]) -> Any:
        """
        Evalue une assignation complete.
        Retourne la valeur combinee de toutes les contraintes.
        """
        values = []
        for vars_involved, evaluator in self.constraints:
            relevant_assignment = {v: assignment[v] for v in vars_involved}
            values.append(evaluator(relevant_assignment))
        
        # Combiner toutes les valeurs avec l'operateur de projection
        result = self.semiring.top
        for v in values:
            result = self.semiring.project(result, v)
        return result
    
    def solve_brute_force(self) -> Tuple[Dict[str, Any], Any]:
        """
        Resolution par enumeration complete.
        Retourne (meilleure_assignation, meilleure_valeur).
        """
        from itertools import product
        
        var_names = list(self.variables.keys())
        domains = [self.variables[v] for v in var_names]
        
        best_assignment = None
        best_value = self.semiring.bottom
        
        for values in product(*domains):
            assignment = dict(zip(var_names, values))
            value = self.evaluate_assignment(assignment)
            
            # Comparer avec l'operateur de combinaison
            if self.semiring.combine(value, best_value) == value and value != best_value:
                best_value = value
                best_assignment = assignment
        
        return best_assignment, best_value

print("Classe SoftCSP definie.")

---

## 3. Fuzzy CSP : Degres d'appartenance (~15 min)

Le **Fuzzy CSP** utilise des degres dans $[0, 1]$ pour chaque contrainte :
- **1.0** : contrainte parfaitement satisfaite
- **0.5** : contrainte partiellement satisfaite
- **0.0** : contrainte completement violee

### Combinaison
La satisfaction globale est le **minimum** des degres de toutes les contraintes (principe du "maillon le plus faible").

### Exemple : Planning de vacances

Variables :
- **Destination** : Paris, Londres, Rome
- **Mois** : Juin, Juillet, Aout

Contraintes floues :
- Preferer juillet (1.0) > aout (0.7) > juin (0.4)
- Preferer Rome (1.0) > Paris (0.8) > Londres (0.3)
- Budget limite (fuzzy sur le prix)

In [ ]:
# Exemple Fuzzy CSP : Planning de vacances

# Fonctions d'appartenance floues
def month_preference(assign):
    """Preference sur le mois de depart."""
    month = assign['month']
    prefs = {'Juillet': 1.0, 'Aout': 0.7, 'Juin': 0.4}
    return prefs.get(month, 0.0)

def destination_preference(assign):
    """Preference sur la destination."""
    dest = assign['destination']
    prefs = {'Rome': 1.0, 'Paris': 0.8, 'Londres': 0.3}
    return prefs.get(dest, 0.0)

def budget_constraint(assign):
    """
    Contrainte budget floue.
    - < 1000 EUR : parfaitement OK (1.0)
    - 1000-1500 EUR : acceptable (decreasing)
    - > 1500 EUR : inacceptable (0.0)
    """
    costs = {
        ('Rome', 'Juillet'): 1400,
        ('Rome', 'Aout'): 1200,
        ('Rome', 'Juin'): 1000,
        ('Paris', 'Juillet'): 1100,
        ('Paris', 'Aout'): 900,
        ('Paris', 'Juin'): 800,
        ('Londres', 'Juillet'): 1000,
        ('Londres', 'Aout'): 800,
        ('Londres', 'Juin'): 700,
    }
    cost = costs.get((assign['destination'], assign['month']), 2000)
    
    if cost <= 1000:
        return 1.0
    elif cost <= 1500:
        return 1.0 - (cost - 1000) / 500  # Lineaire decroissant
    else:
        return 0.0

# Construction du Fuzzy CSP
fuzzy_csp = SoftCSP(FuzzySemiring)
fuzzy_csp.add_variable('destination', ['Paris', 'Londres', 'Rome'])
fuzzy_csp.add_variable('month', ['Juin', 'Juillet', 'Aout'])

fuzzy_csp.add_constraint(['month'], month_preference)
fuzzy_csp.add_constraint(['destination'], destination_preference)
fuzzy_csp.add_constraint(['destination', 'month'], budget_constraint)

# Resolution
best_assign, best_value = fuzzy_csp.solve_brute_force()

print("=== Fuzzy CSP : Planning de Vacances ===")
print(f"Meilleure assignation : {best_assign}")
print(f"Degre de satisfaction global : {best_value:.2f}")

# Afficher toutes les solutions avec leur degre
print("\nToutes les solutions classees par preference :")
from itertools import product
results = []
for values in product(['Paris', 'Londres', 'Rome'], ['Juin', 'Juillet', 'Aout']):
    assignment = {'destination': values[0], 'month': values[1]}
    value = fuzzy_csp.evaluate_assignment(assignment)
    results.append((assignment, value))

results.sort(key=lambda x: -x[1])
for assign, val in results:
    print(f"  {assign} -> satisfaction: {val:.2f}")

In [ ]:
# Visualisation des preferences floues
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Preference par mois
months = ['Juin', 'Juillet', 'Aout']
month_prefs = [0.4, 1.0, 0.7]
axes[0].bar(months, month_prefs, color='steelblue', edgecolor='black')
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('Degre de preference')
axes[0].set_title('Preference par Mois')

# Preference par destination
dests = ['Londres', 'Paris', 'Rome']
dest_prefs = [0.3, 0.8, 1.0]
axes[1].bar(dests, dest_prefs, color='coral', edgecolor='black')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Preference par Destination')

# Budget flou
costs = np.linspace(500, 2000, 100)
budget_membership = [1.0 if c <= 1000 else (max(0, 1 - (c - 1000) / 500) if c <= 1500 else 0) for c in costs]
axes[2].plot(costs, budget_membership, 'g-', linewidth=2)
axes[2].axvline(1000, color='green', linestyle='--', alpha=0.5, label='Seuil ideal')
axes[2].axvline(1500, color='red', linestyle='--', alpha=0.5, label='Seuil max')
axes[2].fill_between(costs, budget_membership, alpha=0.3, color='green')
axes[2].set_xlabel('Cout (EUR)')
axes[2].set_ylabel('Degre d\'acceptabilite')
axes[2].set_title('Fonction d\'Appartenance Budget')
axes[2].legend()
axes[2].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

---

## 4. Weighted CSP : Couts de violation (~15 min)

Le **Weighted CSP** associe un **cout** a chaque violation de contrainte :
- La solution optimale minimise la **somme des couts de violation**
- Utile quand certaines contraintes sont plus importantes que d'autres

### Formulation

Minimiser : $\sum_{c \in C} w_c \cdot v_c(assignment)$

ou :
- $w_c$ : poids de la contrainte $c$
- $v_c$ : fonction de violation (0 si satisfaite, > 0 sinon)

### Exemple : Ordonnancement avec preferences

Une tache peut etre planifiee a differents creneaux, avec des preferences sur l'horaire.

In [ ]:
# Exemple Weighted CSP : Planification de reunion

# Variables : 3 participants, chacun a des disponibilites
# Objectif : trouver un creneau minimisant les couts de violation

def availability_cost(assign):
    """
    Cout de disponibilite pour chaque participant.
    0 = disponible, 1 = peu pratique, 2 = indisponible mais urgent
    """
    slot = assign['slot']
    participant = assign['participant']
    
    # Matrice de couts : [participant][slot]
    costs = {
        'Alice': {'9h': 0, '10h': 0, '11h': 1, '14h': 2, '15h': 2},
        'Bob':   {'9h': 2, '10h': 1, '11h': 0, '14h': 0, '15h': 1},
        'Charlie': {'9h': 1, '10h': 0, '11h': 0, '14h': 1, '15h': 0},
    }
    return costs[participant][slot]

def room_cost(assign):
    """
    Cout de la salle.
    0 = salle ideale, 5 = salle de secours
    """
    slot = assign['slot']
    room_availability = {'9h': 0, '10h': 0, '11h': 0, '14h': 5, '15h': 5}
    return room_availability[slot]

# Construction du Weighted CSP
weighted_csp = SoftCSP(WeightedSemiring)

# Pour simplifier, on cherche le meilleur slot unique pour tous
weighted_csp.add_variable('slot', ['9h', '10h', '11h', '14h', '15h'])

# Contraintes : somme des couts de tous les participants + salle
def total_meeting_cost(assign):
    slot = assign['slot']
    total = 0
    # Couts des participants
    for participant in ['Alice', 'Bob', 'Charlie']:
        total += availability_cost({'slot': slot, 'participant': participant})
    # Cout de la salle
    total += room_cost({'slot': slot})
    return total

weighted_csp.add_constraint(['slot'], total_meeting_cost)

# Resolution
best_slot, best_cost = weighted_csp.solve_brute_force()

print("=== Weighted CSP : Planification de Reunion ===")
print(f"Meilleur creneau : {best_slot}")
print(f"Cout total : {best_cost}")

# Detail par slot
print("\nCouts par creneau :")
for slot in ['9h', '10h', '11h', '14h', '15h']:
    cost = weighted_csp.evaluate_assignment({'slot': slot})
    detail = {
        'Alice': availability_cost({'slot': slot, 'participant': 'Alice'}),
        'Bob': availability_cost({'slot': slot, 'participant': 'Bob'}),
        'Charlie': availability_cost({'slot': slot, 'participant': 'Charlie'}),
        'Salle': room_cost({'slot': slot})
    }
    print(f"  {slot}: cout={cost} | Alice:{detail['Alice']} Bob:{detail['Bob']} Charlie:{detail['Charlie']} Salle:{detail['Salle']}")

In [ ]:
# Visualisation des couts
fig, ax = plt.subplots(figsize=(12, 5))

slots = ['9h', '10h', '11h', '14h', '15h']
participants = ['Alice', 'Bob', 'Charlie']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

x = np.arange(len(slots))
width = 0.25

# Couts par participant
for i, (participant, color) in enumerate(zip(participants, colors)):
    costs = [availability_cost({'slot': s, 'participant': participant}) for s in slots]
    ax.bar(x + i * width, costs, width, label=participant, color=color, edgecolor='black')

# Ajouter cout salle
room_costs = [room_cost({'slot': s}) for s in slots]
ax.bar(x + 3 * width, room_costs, width, label='Salle', color='gray', edgecolor='black')

# Marquer le meilleur slot
best_idx = slots.index(best_slot['slot'])
ax.axvline(x=best_idx + 1.5 * width, color='green', linestyle='--', linewidth=2, label='Meilleur choix')

ax.set_xlabel('Creneau')
ax.set_ylabel('Cout de violation')
ax.set_title('Decomposition des Couts par Creneau')
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(slots)
ax.legend()
ax.set_ylim(0, 6)

plt.tight_layout()
plt.show()

---

## 5. Integration avec OR-Tools CP-SAT (~10 min)

OR-Tools CP-SAT peut gerer les contraintes souples via :
1. **Objectifs multiples** : minimiser la somme des violations
2. **Variables auxiliaires** : encoder les violations comme variables
3. **Soft constraints** : contraintes avec penalites

### Exemple : Nurse Scheduling avec preferences

On ajoute des preferences aux infirmiers (certains postes sont preferes).

In [ ]:
def solve_nurse_scheduling_soft(
    num_nurses: int,
    num_days: int,
    shifts_per_day: int,
    min_nurses_per_shift: int,
    max_shifts_per_nurse: int,
    preferences: Dict[Tuple[int, int, int], int] = None
) -> Dict:
    """
    Nurse Scheduling avec contraintes souples (preferences).
    
    Args:
        preferences: Dict[(nurse, day, shift)] -> penalite (0 = prefere, 5 = neutre, 10 = deteste)
    """
    model = cp_model.CpModel()
    
    # Variables binaires: x[nurse, day, shift]
    shifts = {}
    for n in range(num_nurses):
        for d in range(num_days):
            for s in range(shifts_per_day):
                shifts[(n, d, s)] = model.NewBoolVar(f'x_{n}_{d}_{s}')
    
    # Contraintes HARD
    
    # Couverture minimale
    for d in range(num_days):
        for s in range(shifts_per_day):
            model.Add(sum(shifts[(n, d, s)] for n in range(num_nurses)) >= min_nurses_per_shift)
    
    # Max un poste par infirmier par jour
    for n in range(num_nurses):
        for d in range(num_days):
            model.Add(sum(shifts[(n, d, s)] for s in range(shifts_per_day)) <= 1)
    
    # Max postes par infirmier
    for n in range(num_nurses):
        model.Add(sum(shifts[(n, d, s)] for d in range(num_days) for s in range(shifts_per_day)) <= max_shifts_per_nurse)
    
    # Contraintes SOFT (via objectif)
    
    # Penalite totale pour les preferences
    if preferences is None:
        preferences = {}
    
    penalty_terms = []
    for n in range(num_nurses):
        for d in range(num_days):
            for s in range(shifts_per_day):
                # Penalite par defaut : 5 (neutre)
                penalty = preferences.get((n, d, s), 5)
                penalty_terms.append(penalty * shifts[(n, d, s)])
    
    # Objectif : minimiser la penalite totale
    model.Minimize(sum(penalty_terms))
    
    # Resolution
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 30
    status = solver.Solve(model)
    
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        schedule = []
        total_penalty = 0
        for n in range(num_nurses):
            for d in range(num_days):
                for s in range(shifts_per_day):
                    if solver.Value(shifts[(n, d, s)]) == 1:
                        penalty = preferences.get((n, d, s), 5)
                        total_penalty += penalty
                        schedule.append({
                            'nurse': n,
                            'day': d,
                            'shift': s,
                            'penalty': penalty
                        })
        
        return {
            'schedule': schedule,
            'total_penalty': total_penalty,
            'status': 'OPTIMAL' if status == cp_model.OPTIMAL else 'FEASIBLE'
        }
    
    return {'schedule': [], 'total_penalty': float('inf'), 'status': 'INFEASIBLE'}

print("Fonction solve_nurse_scheduling_soft definie.")

In [ ]:
# Exemple avec preferences

# Preferences : l'infirmier 0 prefere les matins, l'infirmier 1 prefere les nuits
preferences = {}

# Infirmier 0 : aime les matins (shift 0), deteste les nuits (shift 2)
for d in range(7):
    preferences[(0, d, 0)] = 0  # Matin = prefere
    preferences[(0, d, 1)] = 5  # Apres-midi = neutre
    preferences[(0, d, 2)] = 10  # Nuit = deteste

# Infirmier 1 : aime les nuits
for d in range(7):
    preferences[(1, d, 0)] = 8
    preferences[(1, d, 1)] = 5
    preferences[(1, d, 2)] = 0  # Nuit = prefere

# Infirmier 2-5 : neutre

result = solve_nurse_scheduling_soft(
    num_nurses=6,
    num_days=7,
    shifts_per_day=3,
    min_nurses_per_shift=2,
    max_shifts_per_nurse=7,
    preferences=preferences
)

print(f"Status: {result['status']}")
print(f"Penalite totale: {result['total_penalty']}")

# Analyser les preferences respectees
shift_names = ['Matin', 'Apres-midi', 'Nuit']
print("\nPlanning avec preferences :")
for s in result['schedule']:
    if s['nurse'] in [0, 1]:
        print(f"  Infirmier {s['nurse']}, Jour {s['day']}, {shift_names[s['shift']]} (penalite: {s['penalty']})")

---

## 6. Resume et Comparaison

| Type | Semi-anneau | Combinaison | Usage principal |
|------|-------------|-------------|-----------------|
| **CSP Classique** | Booleen | AND | Contraintes obligatoires |
| **Fuzzy CSP** | [0,1] | min | Preferences floues |
| **Weighted CSP** | R+ | + | Couts de violation |
| **Hierarchical** | Niveaux | priorite | Contraintes obligatoires + preferees |

### Points cles

1. **Soft CSP** = etendre CSP binaire vers des degres de satisfaction
2. **Semiring-based** = framework unifie pour tous les types
3. **OR-Tools** = integrer soft constraints via objectifs
4. **Applications** : recommandation, planning avec preferences, negociation

---

## 7. Exercices

### Exercice 1 : Menu Planning Fuzzy
Creez un Fuzzy CSP pour planifier un menu hebdomadaire avec des preferences sur les types de plats (viande, poisson, vegetarien).

### Exercice 2 : Weighted CSP pour Voyage
Modelisez un probleme de planification de voyage avec couts sur le temps de trajet, le prix et le confort.

### Exercice 3 : Extension Nurse Scheduling
Ajoutez des contraintes d'equite (equilibrer les penalites entre infirmiers) au modele OR-Tools.

### Exercice 4 : Hierarchical CSP
Implementez un CSP hierarchique avec 3 niveaux : mandatory > strong > weak.

In [ ]:
# Exercice 3 : Template - Equite dans Nurse Scheduling
def solve_nurse_scheduling_fair(
    num_nurses: int,
    num_days: int,
    shifts_per_day: int,
    min_nurses_per_shift: int,
    max_shifts_per_nurse: int,
    preferences: Dict[Tuple[int, int, int], int] = None,
    fairness_weight: int = 10
) -> Dict:
    """
    TODO: Implementez un modele equitable.
    
    Ajoutez une contrainte d'equite :
    - La penalite maximale par infirmier ne doit pas depasser un seuil
    - OU minimisez l'ecart-type des penalites par infirmier
    """
    # Votre code ici
    pass

# Test
# result = solve_nurse_scheduling_fair(6, 7, 3, 2, 7, preferences)

---

## References

1. **Semiring-based CSP** : Bistarelli, Montanari, Rossi (1997) - "Semiring-based Constraint Solving and Optimization"
2. **Soft Constraints** : [ScienceDirect Topics - Soft Constraint](https://www.sciencedirect.com/topics/computer-science/soft-constraint)
3. **Handbook of CP** : Chapitre sur Soft Constraints (2006)
4. **OR-Tools CP-SAT** : [Documentation](https://developers.google.com/optimization/cp)

### Navigation

- [<< Search-15 CSP-Hybridization](Search-15-CSP-Hybridization.ipynb)
- [Index](../README.md)
- [Search-17 CSP-Temporal >>](Search-17-CSP-Temporal.ipynb)